<a href="https://colab.research.google.com/github/BytePhilosopher/OmniSub2026/blob/main/OmniSub2026_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniSub2026 — VSR Pipeline
**Run cells top to bottom. Set runtime to T4 GPU first.**

In [ ]:
# STEP 1 — Kaggle token + install
import os, subprocess, sys
KAGGLE_API_TOKEN = 'KGAT_ca388df72959eba88cca7df75daaffce'  # kaggle.com → Settings → API → Create New Token
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

for pkg in ['kaggle==2.0.0','hydra-core>=1.3.2','opencv-python>=4.5.5.62',
            'scipy>=1.3.0','scikit-image>=0.13.0','av>=10.0.0','six>=1.16.0',
            'mediapipe','gdown>=4.7.3','transformers','accelerate']:
    subprocess.run([sys.executable,'-m','pip','install','-q',pkg])
subprocess.run(['apt-get','install','-qq','ffmpeg'])

if not os.path.exists('/content/AutoAVSR'):
    subprocess.run(['git','clone','-q',
        'https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages',
        '/content/AutoAVSR'])
print('Done.')

In [ ]:
# STEP 2 — Download competition data
from pathlib import Path
DATA_DIR = Path('/content/omnisub/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
!KAGGLE_API_TOKEN={KAGGLE_API_TOKEN} kaggle competitions download -c omni-sub -p "{DATA_DIR}"
!unzip -q "{DATA_DIR}/omni-sub.zip" -d "{DATA_DIR}"
print(f"Test: {len(list((DATA_DIR/'test').glob('*.mp4')))} | Train: {len(list((DATA_DIR/'train').iterdir()))}")

In [ ]:
# STEP 3 — Load model from Google Drive
from google.colab import drive
drive.mount('/content/drive')
MODEL_DIR = Path('/content/AutoAVSR/benchmarks/LRS3/models/LRS3_V_WER19.1')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ZIP = '/content/drive/MyDrive/Copy of LRS3_V_WER19.1.zip'  # change if different path
if not (MODEL_DIR/'model.pth').exists():
    !cp "{DRIVE_ZIP}" /tmp/model.zip
    !unzip -q /tmp/model.zip -d /tmp/model_ex
    !cp /tmp/model_ex/LRS3_V_WER19.1/model.pth "{MODEL_DIR}/model.pth"
    !cp /tmp/model_ex/LRS3_V_WER19.1/model.json "{MODEL_DIR}/model.json"
    print('Model extracted.')
else:
    print('Model already present.')
!ls -lh "{MODEL_DIR}"

In [ ]:
# STEP 4 — MediaPipe compatibility patch (0.10+ fix)
import mediapipe as mp, urllib.request, numpy as np
SHORT = '/tmp/face_short.tflite'
if not os.path.exists(SHORT):
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite', SHORT)

class _BB:
    def __init__(self,x,y,w,h): self.xmin=x;self.ymin=y;self.width=w;self.height=h
class _KP:
    def __init__(self,x,y): self.x=x;self.y=y
class _LD:
    def __init__(self,b,k): self.relative_bounding_box=b;self.relative_keypoints=k
class _Det:
    def __init__(self,l): self.location_data=l
class _Res:
    def __init__(self,d): self.detections=d
class _FKP:
    def __init__(self,i): self.value=i
class _FD:
    def __init__(self,min_detection_confidence=0.5,model_selection=0):
        self._d=mp.tasks.vision.FaceDetector.create_from_options(
            mp.tasks.vision.FaceDetectorOptions(
                base_options=mp.tasks.BaseOptions(model_asset_path=SHORT),
                running_mode=mp.tasks.vision.RunningMode.IMAGE,
                min_detection_confidence=min_detection_confidence))
    def process(self,f):
        r=self._d.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=f.astype(np.uint8)))
        h,w=f.shape[:2]
        return _Res([_Det(_LD(_BB(d.bounding_box.origin_x/w,d.bounding_box.origin_y/h,d.bounding_box.width/w,d.bounding_box.height/h),[_KP(k.x,k.y) for k in d.keypoints])) for d in r.detections])
    def __enter__(self): return self
    def __exit__(self,*a): pass
class _FDM:
    FaceKeyPoint=_FKP
    def FaceDetection(self,**k): return _FD(**k)
class _S:
    face_detection=_FDM()
mp.solutions=_S()
print('MediaPipe patch applied.')

In [ ]:
# STEP 5 — Load inference pipeline
import sys, torch, os
os.chdir('/content/AutoAVSR')
sys.path.insert(0,'/content/AutoAVSR')
from pipelines.pipeline import InferencePipeline

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Write CTC-only config (no LM needed)
with open('configs/LRS3_nolm.ini','w') as f:
    f.write('[input]\nmodality=video\nv_fps=25\n\n[model]\nv_fps=25\n'
            'model_path=benchmarks/LRS3/models/LRS3_V_WER19.1/model.pth\n'
            'model_conf=benchmarks/LRS3/models/LRS3_V_WER19.1/model.json\n'
            'rnnlm=\nrnnlm_conf=\n\n'
            '[decode]\nbeam_size=20\npenalty=0.0\nmaxlenratio=0.0\nminlenratio=0.0\nctc_weight=0.5\nlm_weight=0.0\n')

try:
    pipeline = InferencePipeline('configs/LRS3_V_WER19.1.ini', device=device, detector='mediapipe', face_track=True)
    print('Loaded with LM.')
except Exception as e:
    print(f'LM failed: {e}\nUsing CTC-only...')
    pipeline = InferencePipeline('configs/LRS3_nolm.ini', device=device, detector='mediapipe', face_track=True)
    print('Loaded CTC-only.')

In [ ]:
# STEP 6 — Run inference on all 49 test videos
import csv
from tqdm.notebook import tqdm
from pathlib import Path

DATA_DIR   = Path('/content/omnisub/data')
TEST_DIR   = DATA_DIR / 'test'
SAMPLE_CSV = DATA_DIR / 'sample_submission.csv'
OUTPUT_CSV = DATA_DIR / 'submission.csv'

test_paths = [row['path'] for row in csv.DictReader(open(SAMPLE_CSV))]
print(f'Inferring {len(test_paths)} videos...')

results = []
for name in tqdm(test_paths):
    path = TEST_DIR / name
    try:
        t = pipeline(str(path), landmarks_filename=None).strip().lower()
    except Exception as e:
        print(f'FAILED {name}: {e}')
        t = ''
    results.append({'path': name, 'transcription': t})
    tqdm.write(f'  {name}: {t[:80]}')
print(f'Done. {len(results)} predictions.')

In [ ]:
# STEP 7 — De-repetition + Grammar correction (VSR-LLM pipeline)
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.notebook import tqdm

def remove_repetitions(text):
    if not text or len(text.split()) < 4: return text
    for n in range(8, 1, -1):
        p = r'\b((?:\w[\w\']*\s+){'+str(n-1)+r'}(?:\w[\w\']*))(\s+\1)+\b'
        c = re.sub(p, r'\1', text, flags=re.IGNORECASE)
        if c != text: text = c
    return text.strip()

print('Loading grammar model...')
tokenizer   = AutoTokenizer.from_pretrained('pszemraj/grammar-synthesis-small')
gram_model  = AutoModelForSeq2SeqLM.from_pretrained(
    'pszemraj/grammar-synthesis-small',
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device).eval()

def grammar_correct(text):
    if not text or len(text.split()) < 3: return text
    try:
        inp = tokenizer(text, return_tensors='pt', max_length=128, truncation=True).to(device)
        with torch.no_grad():
            out = gram_model.generate(**inp, max_new_tokens=min(len(text.split())*2,150), num_beams=4, early_stopping=True)
        c = tokenizer.decode(out[0], skip_special_tokens=True).strip().lower()
        return c if len(c.split()) >= len(text.split())*0.6 else text
    except: return text

cleaned = []
for r in tqdm(results):
    raw   = r['transcription']
    fixed = grammar_correct(remove_repetitions(raw))
    if raw != fixed:
        print(f"  {r['path']}:\n    RAW: {raw[:80]}\n    FIX: {fixed[:80]}")
    cleaned.append({'path': r['path'], 'transcription': fixed})
results = cleaned
print(f'Post-processing done. {len(results)} predictions ready.')

In [ ]:
# STEP 8 — Save & Submit
import pandas as pd
from google.colab import files

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(df.to_string())

!KAGGLE_API_TOKEN={KAGGLE_API_TOKEN} kaggle competitions submit -c omni-sub -f "{OUTPUT_CSV}" -m "VSR-LLM: AutoAVSR + de-repetition + grammar synthesis"
files.download(str(OUTPUT_CSV))
print('Done! https://www.kaggle.com/competitions/omni-sub/submissions')